In [3]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sweetviz as sv
pd.set_option("display.max_columns", None)  # Show all columns
pd.set_option("display.width", 1000)
pd.set_option('display.float_format', '{:.3f}'.format) 

In [4]:
os.chdir('/Users/joudisinjab/Documents/GitHub/capstone_project')

LFS = pd.read_csv("Data/LFS/LFS_Cleaned.csv")
JV = pd.read_csv("Data/JV_cleaned.csv")
CPI = pd.read_csv("Data/CPI_cleaned.csv")
LFS.head()

,REC_NUM,lf_status,province,age_5y_group,gender,marital_stat,education,multiple_jobh,work_lasty,ft_pt_last,main_class_worker,immigration_stat,NAICS_21,NOC_10,NOC_43,weeks_absent,paid_for_absence,usualhrs_main,actualhrs_main,ft_pt_main,usualhrs_all,actualhrs_all,hrs_away,paid_overtime,unpaid_overtime,hrs_overtime,current_tenure,prev_tenure,hourly_wage,union_stat,job_permenancy,estab_size,firm_size,duration_unemp,flow_to_unemp,duration_jobless,student_status,econ_fam_type,final_wt,REF_DATE,Month
0,62081,4,24,10,2,2,2,NaN,1.000,1.000,2.000,3,7.000,7.000,31.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,240.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.000,1.000,5,345,2023-01-01,Jan
1,38908,3,48,10,2,1,2,NaN,2.000,NaN,NaN,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.000,7.000,23.000,1.000,11,61,2024-01-01,Jan
2,38907,4,35,2,1,6,2,NaN,3.000,NaN,NaN,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.000,7,268,2024-01-01,Jan
3,38906,1,35,8,1,1,4,1.000,NaN,NaN,3.000,3,6.000,8.000,35.000,NaN,NaN,60.000,60.000,1.000,60.000,60.000,NaN,NaN,NaN,NaN,240.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000,4,249,2024-01-01,Jan
4,38905,1,48,4,1,6,4,1.000,NaN,NaN,2.000,3,20.000,8.000,35.000,NaN,NaN,40.000,40.000,1.000,40.000,40.000,0.000,0.000,0.000,0.000,33.000,NaN,38.000,3.000,1.000,1.000,1.000,NaN,NaN,NaN,1.000,5,81,2024-01-01,Jan


In [7]:
month_order = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

## Formatting Dates columns
LFS['REF_DATE'] = pd.to_datetime(LFS['REF_DATE'])
JV['REF_DATE'] = pd.to_datetime(JV['REF_DATE'])
CPI["REF_DATE"] = pd.to_datetime(CPI["REF_DATE"])

LFS['Month'] = LFS['REF_DATE'].dt.strftime('%b')
LFS['Month'] = pd.Categorical(LFS['Month'], categories=month_order, ordered=True)

JV['Month'] = JV['REF_DATE'].dt.strftime('%b')
JV['Month'] = pd.Categorical(JV['Month'], categories=month_order, ordered=True)
CPI['Month'] = pd.Categorical(CPI['Month'], categories=month_order, ordered=True)

# Formatting categorical variable types 
ordinal_vars = [
    'lf_status', 'age_5y_group', 'education', 'marital_stat', 'work_lasty',
    'job_permenancy', 'estab_size', 'firm_size', 'student_status', 
]

nominal_vars = [
    'province', 'gender', 'multiple_jobh', 'ft_pt_last', 'main_class_worker', 'immigration_stat', 
    'NAICS_21', 'NOC_10', 'NOC_43', 'ft_pt_main', 'union_stat', 'flow_to_unemp' , 'econ_fam_type', 'paid_for_absence'
]
# Convert ordinal variables
for var in ordinal_vars:
    LFS[var] = LFS[var].astype(pd.Int64Dtype())  # Nullable integer type
    LFS[var] = pd.Categorical(LFS[var], ordered=True)

# Convert nominal variables
for var in nominal_vars:
    LFS[var] = LFS[var].astype(pd.Int64Dtype())  # Nullable integer type
    LFS[var] = LFS[var].astype('category')


## Feature engineering labour force participation rate, unemployment rate, employment rate, and market tightness and merging with 'Total, All industries' Job vacancy data

In [8]:
## Defining employment categories to avoid null values

employed = LFS[LFS['lf_status'].isin([1, 2])]

unemployed = LFS[LFS['lf_status'] == 3]

labour_force = LFS[LFS['lf_status'].isin([1, 2, 3])]

hourly_wage = employed[(employed['hourly_wage'] != 0) & (~employed['hourly_wage'].isna())]

jobless = LFS[(LFS['lf_status'] == 3) & ((LFS['work_lasty'] == 1) | (LFS['work_lasty'] == 2))]

weight_col = 'final_wt'


In [9]:
## Grouping by REF_DATE and summing weights to calculate the total estimated number of employed, unemployed, labour force, and working-age population
monthly_employed = employed.groupby('REF_DATE')[weight_col].sum()

monthly_unemployed = unemployed.groupby('REF_DATE')[weight_col].sum()

monthly_labour_force = labour_force.groupby('REF_DATE')[weight_col].sum()

monthly_working_age_pop = LFS.groupby('REF_DATE')[weight_col].sum()

## Grouping by REF_DATE and calculating the weighted sum of hourly wages across employed individuals
weighted_hourly_wage_sum = hourly_wage.groupby('REF_DATE')["hourly_wage"].apply(
    lambda x: (x * hourly_wage.loc[x.index, weight_col]).sum()
)

## Grouping by REF_DATE and summing weights to get the total weighted population of hourly wage earners
total_weight = hourly_wage.groupby('REF_DATE')[weight_col].sum()

## Grouping by REF_DATE and calculating the weighted sum of jobless duration for unemployed individuals who have previously worked
weighted_jobless_duration_sum = jobless.groupby("REF_DATE")["duration_jobless"].apply(
    lambda x: (x * jobless.loc[x.index, "final_wt"]).sum()
)

## Grouping by REF_DATE and summing weights to get the total weighted population of jobless individuals
total_weight_jobless = jobless.groupby("REF_DATE")["final_wt"].sum()


In [10]:
## Calculating Weighted Average Hourly Wage 
average_hourly_wage = weighted_hourly_wage_sum / total_weight

## Calculating Weighted Average Jobless Duration
weighted_jobless_duration = weighted_jobless_duration_sum / total_weight_jobless

## Calculating Monthly Employment & Unemployment Rates & Labour Force Participation Rate
employment_rate = (monthly_employed / monthly_working_age_pop) * 100
unemployment_rate = (monthly_unemployed / monthly_labour_force) * 100
participation_rate = (monthly_labour_force / monthly_working_age_pop) * 100

## Combining into a DataFrame
monthly_rates = pd.DataFrame({
    'Employment_Rate(%)': employment_rate,
    'Unemployment_Rate(%)': unemployment_rate, 
    'Labour_Force_Participation_Rate(%)': participation_rate,
    'Unemployed': monthly_unemployed,
    'Average_Hourly_Wage': average_hourly_wage, 
    'Duration_Jobless(Months)': weighted_jobless_duration
}).reset_index()

In [11]:
## Ensuring REF_DATE is datetime for merge compatibility
monthly_rates['REF_DATE'] = pd.to_datetime(monthly_rates['REF_DATE'])

## Merging with JV & CPI data 
total_vacancies = JV[JV['NAICS_combined'] == 'Total, all industries']
monthly_rates = monthly_rates.merge(total_vacancies, on= 'REF_DATE', how='left')
monthly_rates = monthly_rates.merge(CPI, on= 'REF_DATE', how='left')

## Calculating Market Tightness 
monthly_rates["Market_Tightness"] = monthly_rates["Job Vacancies (#)"] / monthly_rates["Unemployed"]

In [12]:
## Sorting by date
monthly_rates.sort_values(by='REF_DATE', inplace=True)

# Feature engineering lagged Job Vacancy Rate & Unemployment Rate by 1 month
monthly_rates["Lagged_1M_Job_Vacancy_Rate"] = (
    monthly_rates["Job Vacancy Rate (%)"].shift(1))

monthly_rates["Lagged_1M_Unemployment_Rate"] = (
    monthly_rates["Unemployment_Rate(%)"].shift(1))

In [13]:
## Droping redundant NAICS_combined column
monthly_rates.drop(columns=["NAICS_combined", "Unemployed"], inplace=True)

## Removing rows where Year == 2020
monthly_rates = monthly_rates[monthly_rates["Year"] != 2020]

## Extracting monthly rates for 2025 to use for later forecasting 
monthly_2025 = monthly_rates[monthly_rates['Year'] == 2025].copy()

## Dropping rows with missing values
monthly_rates.dropna(inplace=True)

## Exporting working dataset to CSV
monthly_rates.to_csv('Monthly Labour Market Rates.csv', index=False)

In [14]:
## Inspecting the first few rows of the dataset
monthly_rates.head()

,REF_DATE,Employment_Rate(%),Unemployment_Rate(%),Labour_Force_Participation_Rate(%),Average_Hourly_Wage,Duration_Jobless(Months),Job Vacancies (#),Payroll Employees (#),Job Vacancy Rate (%),Month_x,Month_y,Year,All-items,Market_Tightness,Lagged_1M_Job_Vacancy_Rate,Lagged_1M_Unemployment_Rate
4,2021-01-01,57.793,9.841,64.101,30.752,11.519,561280.000,15294425.000,3.500,Jan,Jan,2021,139.100,0.285,3.600,8.188
5,2021-02-01,58.838,8.792,64.510,30.639,13.636,616555.000,15167450.000,3.900,Feb,Feb,2021,139.400,0.348,3.500,9.841
6,2021-03-01,59.615,8.205,64.944,30.484,13.550,643655.000,15233105.000,4.100,Mar,Mar,2021,139.500,0.387,3.900,8.792
7,2021-04-01,59.234,8.455,64.705,30.742,13.077,651890.000,15431950.000,4.100,Apr,Apr,2021,139.900,0.381,4.100,8.205
8,2021-05-01,60.141,8.392,65.650,30.549,14.394,673585.000,15556570.000,4.200,May,May,2021,140.300,0.391,4.100,8.455


In [15]:
## Inspecting the first few rows of the dataset
monthly_2025.head()

,REF_DATE,Employment_Rate(%),Unemployment_Rate(%),Labour_Force_Participation_Rate(%),Average_Hourly_Wage,Duration_Jobless(Months),Job Vacancies (#),Payroll Employees (#),Job Vacancy Rate (%),Month_x,Month_y,Year,All-items,Market_Tightness,Lagged_1M_Job_Vacancy_Rate,Lagged_1M_Unemployment_Rate
52,2025-01-01,60.191,7.050,64.757,35.995,15.819,NaN,NaN,NaN,NaN,Jan,2025,162.600,NaN,3.000,6.166
53,2025-02-01,60.395,6.686,64.722,36.137,18.400,NaN,NaN,NaN,NaN,Feb,2025,163.700,NaN,NaN,7.050


## Feature engineering labour force participation rate, unemployment rate, employment rate, and market tightness by industry and merging with Job vacancy and CPI data

In [16]:
# Grouping Employed (LF_STATUS: 1, 2) by month & industry
employed_by_month_industry = (
    LFS[LFS['lf_status'].isin([1, 2])]
    .groupby(['REF_DATE', 'NAICS_21'], observed=False)[weight_col]
    .sum()
    .reset_index()
    .rename(columns={weight_col: 'total_employed'})
)

# Grouping Unemployed (LF_STATUS: 3) by month & industry
unemployed_by_month_industry = (
    LFS[LFS['lf_status'] == 3]
    .groupby(['REF_DATE', 'NAICS_21'], observed=False)[weight_col]
    .sum()
    .reset_index()
    .rename(columns={weight_col: 'total_unemployed'})
)

# Grouping Labour Force Participation
participation_by_month_industry = (
    LFS[LFS['lf_status'].isin([1, 2, 3])]
    .groupby(['REF_DATE', 'NAICS_21'], observed=False)[weight_col]
    .sum()
    .reset_index()
    .rename(columns={weight_col: 'total_participation'})
)

# Grouping Working Age Population
working_age_by_month_industry = (
    LFS
    .groupby(['REF_DATE', 'NAICS_21'], observed=False)[weight_col]
    .sum()
    .reset_index()
    .rename(columns={weight_col: 'total_working_age'})
)

# Grouping Duration Jobless
jobless_industry = LFS[(LFS['lf_status'] == 3) & ((LFS['work_lasty'] == 1) | (LFS['work_lasty'] == 2))]

weighted_jobless_duration_sum = (
    jobless_industry
    .groupby(['REF_DATE', 'NAICS_21'], observed=False)[["duration_jobless", weight_col]] 
    .apply(lambda x: (x["duration_jobless"] * x[weight_col]).sum())
    .reset_index(name="weighted_jobless_duration_sum")
)

# Compute Total Weights per Industry & Month
total_weight_jobless = (
    jobless_industry.groupby(["REF_DATE", "NAICS_21"], observed=False)["final_wt"]
    .sum()
    .reset_index(name="total_weight_jobless")
)

# Grouping Hourly Wage
hourly_wage = LFS[(LFS['lf_status'].isin([1, 2])) & (LFS['hourly_wage'] != 0) & (~LFS['hourly_wage'].isna())]

weighted_hourly_wage_sum = (
    hourly_wage
    .groupby(['REF_DATE', 'NAICS_21'], observed=False)[["hourly_wage", weight_col]]  
    .apply(lambda x: (x["hourly_wage"] * x[weight_col]).sum())
    .reset_index(name="weighted_hourly_wage_sum")
)



# Compute Total Weights per Industry & Month
total_weight_hourly_wage = (
    hourly_wage.groupby(["REF_DATE", "NAICS_21"], observed=False)[weight_col]
    .sum()
    .reset_index(name="total_weight_hourly_wage")
)

# Merge all employment-related metrics
monthly_totals = (
    employed_by_month_industry
    .merge(unemployed_by_month_industry, on=['REF_DATE', 'NAICS_21'], how='outer')
    .merge(participation_by_month_industry, on=['REF_DATE', 'NAICS_21'], how='outer')
    .merge(working_age_by_month_industry, on=['REF_DATE', 'NAICS_21'], how='outer')
    .merge(total_weight_jobless, on=['REF_DATE', 'NAICS_21'], how='outer')
    .merge(weighted_jobless_duration_sum, on=['REF_DATE', 'NAICS_21'], how='outer')
    .merge(total_weight_hourly_wage, on=['REF_DATE', 'NAICS_21'], how='outer')
    .merge(weighted_hourly_wage_sum, on=['REF_DATE', 'NAICS_21'], how='outer')
)

# Check for missing values
print(monthly_totals.isna().sum())

# Display first few rows
monthly_totals.head()


REF_DATE                         0
NAICS_21                         0
total_employed                   0
total_unemployed                 0
total_participation              0
total_working_age                0
total_weight_jobless             0
weighted_jobless_duration_sum    0
total_weight_hourly_wage         0
weighted_hourly_wage_sum         0
dtype: int64


,REF_DATE,NAICS_21,total_employed,total_unemployed,total_participation,total_working_age,total_weight_jobless,weighted_jobless_duration_sum,total_weight_hourly_wage,weighted_hourly_wage_sum
0,2020-09-01,1,286125,9707,295832,323708,9707,47850.000,123044,2554083.130
1,2020-09-01,2,55231,5275,60506,69826,5275,19680.000,45416,1357742.900
2,2020-09-01,3,16828,6238,23066,31566,6238,23079.000,7950,185220.540
3,2020-09-01,4,225784,27257,253041,267383,27257,159029.000,215833,10013609.180
4,2020-09-01,5,140835,1689,142524,146054,1689,7610.000,140835,6486970.550


In [17]:
# Check for missing values
print(monthly_totals.isna().sum())

# Display first few rows
monthly_totals.head()

REF_DATE                         0
NAICS_21                         0
total_employed                   0
total_unemployed                 0
total_participation              0
total_working_age                0
total_weight_jobless             0
weighted_jobless_duration_sum    0
total_weight_hourly_wage         0
weighted_hourly_wage_sum         0
dtype: int64


,REF_DATE,NAICS_21,total_employed,total_unemployed,total_participation,total_working_age,total_weight_jobless,weighted_jobless_duration_sum,total_weight_hourly_wage,weighted_hourly_wage_sum
0,2020-09-01,1,286125,9707,295832,323708,9707,47850.000,123044,2554083.130
1,2020-09-01,2,55231,5275,60506,69826,5275,19680.000,45416,1357742.900
2,2020-09-01,3,16828,6238,23066,31566,6238,23079.000,7950,185220.540
3,2020-09-01,4,225784,27257,253041,267383,27257,159029.000,215833,10013609.180
4,2020-09-01,5,140835,1689,142524,146054,1689,7610.000,140835,6486970.550


In [18]:
# Ensure REF_DATE is in datetime format
monthly_totals["REF_DATE"] = pd.to_datetime(monthly_totals["REF_DATE"])
JV["REF_DATE"] = pd.to_datetime(JV["REF_DATE"])
CPI["REF_DATE"] = pd.to_datetime(CPI["REF_DATE"])

# Drop "Total, all industries" from JV dataset
JV = JV[JV['NAICS_combined'] != 'Total, all industries']

# Map JV industries to numerical categories
industry_encoding = {
    'Accommodation and food services [72]': 1, 
    'Business, building and other support services [55-56]': 2, 
    'Agriculture, forestry, fishing and hunting [11]': 3, 
    'Construction [23]': 4,
    'Educational services [61]': 5, 
    'Finance and insurance [52]': 6,
    'Health care and social assistance [62]': 7,
    'Information, culture and recreation [51-71]': 8, 
    'Manufacturing [31-33]': 9, 
    'Mining, quarrying, and oil and gas extraction [21]': 10, 
    'Other services (except public administration) [81]': 11,
    'Professional, scientific and technical services [54]': 12,
    'Public administration [91]': 13,
    'Real estate and rental and leasing [53]': 14,
    'Retail trade [44-45]': 15,
    'Transportation and warehousing [48-49]': 16,
    'Utilities [22]': 17,
    'Wholesale trade [41]': 18
}

JV["Industry_Label"] = JV["NAICS_combined"]
JV['JV_industry'] = JV['NAICS_combined'].map(industry_encoding)

if JV['JV_industry'].isnull().any():
    print("Warning: Some industries could not be mapped. Check for missing values.")

# Mapping LFS to JV industry codes
LFS_to_JV = {
    1 : 3,
    2 : 3,  
    3 : 3,  
    4 : 10,
    5 : 17,
    6 : 4,
    7 : 9,  
    8 : 9,  
    9 : 18,
    10: 15,
    11: 16,
    12: 6,
    13: 14,
    14: 12,
    15: 2,  
    16: 5,
    17: 7,
    18: 8,  
    19: 1,
    20: 11,
    21: 13,
}

monthly_totals['NAICS_21_mapped'] = monthly_totals['NAICS_21'].map(LFS_to_JV)

# Check for missing mappings
if monthly_totals['NAICS_21_mapped'].isnull().any():
    print("Warning: Some industries could not be mapped. Check missing values:", 
          monthly_totals[monthly_totals['NAICS_21_mapped'].isnull()]['NAICS_21'].unique())

# Merging JV & CPI with industry monthly rates
monthly_totals = monthly_totals.merge(
    JV[['REF_DATE', 'JV_industry', 'Industry_Label', 'Job Vacancies (#)', 'Payroll Employees (#)', 'Job Vacancy Rate (%)']], 
    left_on=["REF_DATE", "NAICS_21_mapped"], 
    right_on=["REF_DATE", "JV_industry"], 
    how="left"
)


monthly_totals.dropna(inplace=True)
# Display first few rows
monthly_totals.head()


/var/folders/n9/29yl9jnj61q74vg3ydjdf3qh0000gn/T/ipykernel_26907/2716309161.py:31: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  JV["Industry_Label"] = JV["NAICS_combined"]
/var/folders/n9/29yl9jnj61q74vg3ydjdf3qh0000gn/T/ipykernel_26907/2716309161.py:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  JV['JV_industry'] = JV['NAICS_combined'].map(industry_encoding)


,REF_DATE,NAICS_21,total_employed,total_unemployed,total_participation,total_working_age,total_weight_jobless,weighted_jobless_duration_sum,total_weight_hourly_wage,weighted_hourly_wage_sum,NAICS_21_mapped,JV_industry,Industry_Label,Job Vacancies (#),Payroll Employees (#),Job Vacancy Rate (%)
21,2020-10-01,1,287401,13462,300863,330035,13462,57446.000,128688,2674172.340,3,3.000,"Agriculture, forestry, fishing and hunting [11]",16925.000,233620.000,6.800
22,2020-10-01,2,44657,4807,49464,57869,4807,23125.000,35327,1080353.060,3,3.000,"Agriculture, forestry, fishing and hunting [11]",16925.000,233620.000,6.800
23,2020-10-01,3,16436,6820,23256,32741,6820,32261.000,4183,113228.340,3,3.000,"Agriculture, forestry, fishing and hunting [11]",16925.000,233620.000,6.800
24,2020-10-01,4,248627,24772,273399,286151,24772,172005.000,236330,10721868.230,10,10.000,"Mining, quarrying, and oil and gas extraction ...",4195.000,183335.000,2.200
25,2020-10-01,5,138456,1107,139563,142207,1107,5140.000,138456,6397023.410,17,17.000,Utilities [22],1420.000,122230.000,1.100


Agriculture, forestry, fishing and hunting [11] and Manufacturing [31-33] are each one industry in JV. However, in LFS the Manufacturing [31-33] is divided into 2 sub-industries and Agriculture, forestry, fishing and hunting [11] is divided into 3 sub-industries. The job vacancy data needs to be proportional to each industry. 

In [19]:
# Select industries needing adjustment
monthly_totals['NAICS_21'] = monthly_totals['NAICS_21'].astype('category')
monthly_totals['NAICS_21_mapped'] = monthly_totals['NAICS_21_mapped'].astype('category')

In [20]:
# Select industries needing adjustment
adjusted_rates = monthly_totals[monthly_totals['NAICS_21'].isin([1,2,3,7,8])]
adjusted_rates.head()


,REF_DATE,NAICS_21,total_employed,total_unemployed,total_participation,total_working_age,total_weight_jobless,weighted_jobless_duration_sum,total_weight_hourly_wage,weighted_hourly_wage_sum,NAICS_21_mapped,JV_industry,Industry_Label,Job Vacancies (#),Payroll Employees (#),Job Vacancy Rate (%)
21,2020-10-01,1,287401,13462,300863,330035,13462,57446.000,128688,2674172.340,3,3.000,"Agriculture, forestry, fishing and hunting [11]",16925.000,233620.000,6.800
22,2020-10-01,2,44657,4807,49464,57869,4807,23125.000,35327,1080353.060,3,3.000,"Agriculture, forestry, fishing and hunting [11]",16925.000,233620.000,6.800
23,2020-10-01,3,16436,6820,23256,32741,6820,32261.000,4183,113228.340,3,3.000,"Agriculture, forestry, fishing and hunting [11]",16925.000,233620.000,6.800
27,2020-10-01,7,904675,48068,952743,998961,48068,235426.000,864203,26401742.560,9,9.000,Manufacturing [31-33],46345.000,1474900.000,3.000
28,2020-10-01,8,832177,48999,881176,934374,48999,279578.000,788747,21725393.540,9,9.000,Manufacturing [31-33],46345.000,1474900.000,3.000


In [21]:
total_weighted_labour_force_per_JV = (
    adjusted_rates.groupby(['REF_DATE', 'NAICS_21_mapped'], observed=False)['total_participation']
    .sum()
    .reset_index()
    .rename(columns={'total_participation': 'total_weighted_labour_force_JV'})
)
print(total_weighted_labour_force_per_JV.head())
# Merge total weighted labour force per JV industry
adjusted_rates = adjusted_rates.merge(total_weighted_labour_force_per_JV, on=['REF_DATE', 'NAICS_21_mapped'], how='left')
adjusted_rates.head()

    REF_DATE NAICS_21_mapped  total_weighted_labour_force_JV
0 2020-10-01               1                               0
1 2020-10-01               2                               0
2 2020-10-01               3                          373583
3 2020-10-01               4                               0
4 2020-10-01               5                               0


,REF_DATE,NAICS_21,total_employed,total_unemployed,total_participation,total_working_age,total_weight_jobless,weighted_jobless_duration_sum,total_weight_hourly_wage,weighted_hourly_wage_sum,NAICS_21_mapped,JV_industry,Industry_Label,Job Vacancies (#),Payroll Employees (#),Job Vacancy Rate (%),total_weighted_labour_force_JV
0,2020-10-01,1,287401,13462,300863,330035,13462,57446.000,128688,2674172.340,3,3.000,"Agriculture, forestry, fishing and hunting [11]",16925.000,233620.000,6.800,373583
1,2020-10-01,2,44657,4807,49464,57869,4807,23125.000,35327,1080353.060,3,3.000,"Agriculture, forestry, fishing and hunting [11]",16925.000,233620.000,6.800,373583
2,2020-10-01,3,16436,6820,23256,32741,6820,32261.000,4183,113228.340,3,3.000,"Agriculture, forestry, fishing and hunting [11]",16925.000,233620.000,6.800,373583
3,2020-10-01,7,904675,48068,952743,998961,48068,235426.000,864203,26401742.560,9,9.000,Manufacturing [31-33],46345.000,1474900.000,3.000,1833919
4,2020-10-01,8,832177,48999,881176,934374,48999,279578.000,788747,21725393.540,9,9.000,Manufacturing [31-33],46345.000,1474900.000,3.000,1833919


In [22]:
# Compute labour force proportion using weighted values
adjusted_rates['labour_force_proportion'] = adjusted_rates['total_participation'] / adjusted_rates['total_weighted_labour_force_JV']
adjusted_rates['labour_force_proportion'].fillna(0, inplace=True)  # Replace NaN proportions with 0
adjusted_rates.head()   

/var/folders/n9/29yl9jnj61q74vg3ydjdf3qh0000gn/T/ipykernel_26907/4265945580.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  adjusted_rates['labour_force_proportion'].fillna(0, inplace=True)  # Replace NaN proportions with 0


,REF_DATE,NAICS_21,total_employed,total_unemployed,total_participation,total_working_age,total_weight_jobless,weighted_jobless_duration_sum,total_weight_hourly_wage,weighted_hourly_wage_sum,NAICS_21_mapped,JV_industry,Industry_Label,Job Vacancies (#),Payroll Employees (#),Job Vacancy Rate (%),total_weighted_labour_force_JV,labour_force_proportion
0,2020-10-01,1,287401,13462,300863,330035,13462,57446.000,128688,2674172.340,3,3.000,"Agriculture, forestry, fishing and hunting [11]",16925.000,233620.000,6.800,373583,0.805
1,2020-10-01,2,44657,4807,49464,57869,4807,23125.000,35327,1080353.060,3,3.000,"Agriculture, forestry, fishing and hunting [11]",16925.000,233620.000,6.800,373583,0.132
2,2020-10-01,3,16436,6820,23256,32741,6820,32261.000,4183,113228.340,3,3.000,"Agriculture, forestry, fishing and hunting [11]",16925.000,233620.000,6.800,373583,0.062
3,2020-10-01,7,904675,48068,952743,998961,48068,235426.000,864203,26401742.560,9,9.000,Manufacturing [31-33],46345.000,1474900.000,3.000,1833919,0.520
4,2020-10-01,8,832177,48999,881176,934374,48999,279578.000,788747,21725393.540,9,9.000,Manufacturing [31-33],46345.000,1474900.000,3.000,1833919,0.480


In [23]:
# Adjust Job Vacancies and Payroll Employees based on labour force proportion
adjusted_rates['Adjusted Job Vacancies'] = (
    adjusted_rates['Job Vacancies (#)'] * adjusted_rates['labour_force_proportion']
).round(0)

adjusted_rates['Adjusted Payroll Employees'] = (
    adjusted_rates['Payroll Employees (#)'] * adjusted_rates['labour_force_proportion']
).round(0)

# Compute adjusted Job Vacancy Rate (%)
adjusted_rates['Adjusted Job Vacancy Rate (%)'] = (
    adjusted_rates['Adjusted Job Vacancies'] /
    (adjusted_rates['Adjusted Payroll Employees'] + adjusted_rates['Adjusted Job Vacancies'])
) * 100


adjusted_rates.head()

,REF_DATE,NAICS_21,total_employed,total_unemployed,total_participation,total_working_age,total_weight_jobless,weighted_jobless_duration_sum,total_weight_hourly_wage,weighted_hourly_wage_sum,NAICS_21_mapped,JV_industry,Industry_Label,Job Vacancies (#),Payroll Employees (#),Job Vacancy Rate (%),total_weighted_labour_force_JV,labour_force_proportion,Adjusted Job Vacancies,Adjusted Payroll Employees,Adjusted Job Vacancy Rate (%)
0,2020-10-01,1,287401,13462,300863,330035,13462,57446.000,128688,2674172.340,3,3.000,"Agriculture, forestry, fishing and hunting [11]",16925.000,233620.000,6.800,373583,0.805,13630.000,188145.000,6.755
1,2020-10-01,2,44657,4807,49464,57869,4807,23125.000,35327,1080353.060,3,3.000,"Agriculture, forestry, fishing and hunting [11]",16925.000,233620.000,6.800,373583,0.132,2241.000,30932.000,6.755
2,2020-10-01,3,16436,6820,23256,32741,6820,32261.000,4183,113228.340,3,3.000,"Agriculture, forestry, fishing and hunting [11]",16925.000,233620.000,6.800,373583,0.062,1054.000,14543.000,6.758
3,2020-10-01,7,904675,48068,952743,998961,48068,235426.000,864203,26401742.560,9,9.000,Manufacturing [31-33],46345.000,1474900.000,3.000,1833919,0.520,24077.000,766228.000,3.047
4,2020-10-01,8,832177,48999,881176,934374,48999,279578.000,788747,21725393.540,9,9.000,Manufacturing [31-33],46345.000,1474900.000,3.000,1833919,0.480,22268.000,708672.000,3.046


In [24]:

# Drop old columns and rename adjusted values
adjusted_rates.drop(columns=['Job Vacancies (#)', 'Payroll Employees (#)', 'Job Vacancy Rate (%)'], inplace=True)
adjusted_rates.rename(columns=
                      {'Adjusted Job Vacancies': 'Job Vacancies (#)', 
                       'Adjusted Payroll Employees': 'Payroll Employees (#)', 
                       'Adjusted Job Vacancy Rate (%)': 'Job Vacancy Rate (%)'}, inplace=True)

# Ensure industries that need adjustment are removed from main dataset
industries_to_update = [1, 2, 3, 7, 8]
monthly_totals = monthly_totals[~monthly_totals['NAICS_21'].isin(industries_to_update)]

# Append adjusted data back into dataset
monthly_industry_rates = pd.concat([monthly_totals, adjusted_rates], ignore_index=True)

# Ensure data is sorted correctly
monthly_industry_rates = monthly_industry_rates.sort_values(by=['REF_DATE', 'NAICS_21']).reset_index(drop=True)

monthly_industry_rates.head()

,REF_DATE,NAICS_21,total_employed,total_unemployed,total_participation,total_working_age,total_weight_jobless,weighted_jobless_duration_sum,total_weight_hourly_wage,weighted_hourly_wage_sum,NAICS_21_mapped,JV_industry,Industry_Label,Job Vacancies (#),Payroll Employees (#),Job Vacancy Rate (%),total_weighted_labour_force_JV,labour_force_proportion
0,2020-10-01,1,287401,13462,300863,330035,13462,57446.000,128688,2674172.340,3,3.000,"Agriculture, forestry, fishing and hunting [11]",13630.000,188145.000,6.755,373583.000,0.805
1,2020-10-01,2,44657,4807,49464,57869,4807,23125.000,35327,1080353.060,3,3.000,"Agriculture, forestry, fishing and hunting [11]",2241.000,30932.000,6.755,373583.000,0.132
2,2020-10-01,3,16436,6820,23256,32741,6820,32261.000,4183,113228.340,3,3.000,"Agriculture, forestry, fishing and hunting [11]",1054.000,14543.000,6.758,373583.000,0.062
3,2020-10-01,4,248627,24772,273399,286151,24772,172005.000,236330,10721868.230,10,10.000,"Mining, quarrying, and oil and gas extraction ...",4195.000,183335.000,2.200,NaN,NaN
4,2020-10-01,5,138456,1107,139563,142207,1107,5140.000,138456,6397023.410,17,17.000,Utilities [22],1420.000,122230.000,1.100,NaN,NaN


In [25]:
# Calculate labour market indicators by weight
monthly_industry_rates["Employment_Rate(%)"] = (monthly_industry_rates["total_employed"] / monthly_industry_rates["total_working_age"]) * 100
monthly_industry_rates["Unemployment_Rate(%)"] = (monthly_industry_rates["total_unemployed"] / monthly_industry_rates["total_participation"]) * 100
monthly_industry_rates["Labour_Force_Participation_Rate(%)"] = (monthly_industry_rates["total_participation"] / monthly_industry_rates["total_working_age"]) * 100
monthly_industry_rates["Duration_Jobless(Months)"] = monthly_industry_rates["weighted_jobless_duration_sum"] / monthly_industry_rates["total_weight_jobless"]
monthly_industry_rates["Average_Hourly_Wage"] = monthly_industry_rates["weighted_hourly_wage_sum"] / monthly_industry_rates["total_weight_hourly_wage"]
monthly_industry_rates["Market_Tightness"] = monthly_industry_rates["Job Vacancies (#)"] / monthly_industry_rates["total_unemployed"]

monthly_industry_rates.head()

,REF_DATE,NAICS_21,total_employed,total_unemployed,total_participation,total_working_age,total_weight_jobless,weighted_jobless_duration_sum,total_weight_hourly_wage,weighted_hourly_wage_sum,NAICS_21_mapped,JV_industry,Industry_Label,Job Vacancies (#),Payroll Employees (#),Job Vacancy Rate (%),total_weighted_labour_force_JV,labour_force_proportion,Employment_Rate(%),Unemployment_Rate(%),Labour_Force_Participation_Rate(%),Duration_Jobless(Months),Average_Hourly_Wage,Market_Tightness
0,2020-10-01,1,287401,13462,300863,330035,13462,57446.000,128688,2674172.340,3,3.000,"Agriculture, forestry, fishing and hunting [11]",13630.000,188145.000,6.755,373583.000,0.805,87.082,4.474,91.161,4.267,20.780,1.012
1,2020-10-01,2,44657,4807,49464,57869,4807,23125.000,35327,1080353.060,3,3.000,"Agriculture, forestry, fishing and hunting [11]",2241.000,30932.000,6.755,373583.000,0.132,77.169,9.718,85.476,4.811,30.582,0.466
2,2020-10-01,3,16436,6820,23256,32741,6820,32261.000,4183,113228.340,3,3.000,"Agriculture, forestry, fishing and hunting [11]",1054.000,14543.000,6.758,373583.000,0.062,50.200,29.326,71.030,4.730,27.069,0.155
3,2020-10-01,4,248627,24772,273399,286151,24772,172005.000,236330,10721868.230,10,10.000,"Mining, quarrying, and oil and gas extraction ...",4195.000,183335.000,2.200,NaN,NaN,86.887,9.061,95.544,6.944,45.368,0.169
4,2020-10-01,5,138456,1107,139563,142207,1107,5140.000,138456,6397023.410,17,17.000,Utilities [22],1420.000,122230.000,1.100,NaN,NaN,97.362,0.793,98.141,4.643,46.203,1.283


In [26]:
naics_21_labels = {
    1: "Agriculture",
    2: "Forestry and logging and support activities for forestry",
    3: "Fishing, hunting and trapping",
    4: "Mining, quarrying, and oil and gas extraction",
    5: "Utilities",
    6: "Construction",
    7: "Manufacturing - durable goods",
    8: "Manufacturing - non-durable goods",
    9: "Wholesale trade",
    10: "Retail trade",
    11: "Transportation and warehousing",
    12: "Finance and insurance",
    13: "Real estate and rental and leasing",
    14: "Professional, scientific and technical services",
    15: "Business, building and other support services",
    16: "Educational services",
    17: "Health care and social assistance",
    18: "Information, culture and recreation",
    19: "Accommodation and food services",
    20: "Other services (except public administration)",
    21: "Public administration"
}

monthly_industry_rates["NAICS_21_Label"] = monthly_industry_rates["NAICS_21"].map(naics_21_labels)


In [27]:
monthly_industry_rates.drop(columns= 
                            ['total_employed', 'total_unemployed', 'total_participation', 'total_working_age', 
                             'total_weight_jobless', 'weighted_jobless_duration_sum', 'total_weight_hourly_wage', 
                             'weighted_hourly_wage_sum', 'labour_force_proportion',
                             'NAICS_21_mapped', 'total_weighted_labour_force_JV'], inplace=True)
monthly_industry_rates.head()

,REF_DATE,NAICS_21,JV_industry,Industry_Label,Job Vacancies (#),Payroll Employees (#),Job Vacancy Rate (%),Employment_Rate(%),Unemployment_Rate(%),Labour_Force_Participation_Rate(%),Duration_Jobless(Months),Average_Hourly_Wage,Market_Tightness,NAICS_21_Label
0,2020-10-01,1,3.000,"Agriculture, forestry, fishing and hunting [11]",13630.000,188145.000,6.755,87.082,4.474,91.161,4.267,20.780,1.012,Agriculture
1,2020-10-01,2,3.000,"Agriculture, forestry, fishing and hunting [11]",2241.000,30932.000,6.755,77.169,9.718,85.476,4.811,30.582,0.466,Forestry and logging and support activities fo...
2,2020-10-01,3,3.000,"Agriculture, forestry, fishing and hunting [11]",1054.000,14543.000,6.758,50.200,29.326,71.030,4.730,27.069,0.155,"Fishing, hunting and trapping"
3,2020-10-01,4,10.000,"Mining, quarrying, and oil and gas extraction ...",4195.000,183335.000,2.200,86.887,9.061,95.544,6.944,45.368,0.169,"Mining, quarrying, and oil and gas extraction"
4,2020-10-01,5,17.000,Utilities [22],1420.000,122230.000,1.100,97.362,0.793,98.141,4.643,46.203,1.283,Utilities


In [28]:
# Lag Job Vacancy Rate & Unemployment Rate by 1 month
monthly_industry_rates["Lagged_Job_Vacancy_Rate"] = (
    monthly_industry_rates
    .groupby("NAICS_21")["Job Vacancy Rate (%)"]
    .shift(1)
)

monthly_industry_rates["Lagged_Unemployment_Rate"] = (
    monthly_industry_rates
    .groupby("NAICS_21")["Unemployment_Rate(%)"]
    .shift(1)
)

monthly_industry_rates["Lagged_Market_Tightness"] = (
    monthly_industry_rates
    .groupby("NAICS_21")["Market_Tightness"]
    .shift(1)
)
monthly_industry_rates.head()

/var/folders/n9/29yl9jnj61q74vg3ydjdf3qh0000gn/T/ipykernel_26907/3123976544.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby("NAICS_21")["Job Vacancy Rate (%)"]
/var/folders/n9/29yl9jnj61q74vg3ydjdf3qh0000gn/T/ipykernel_26907/3123976544.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby("NAICS_21")["Unemployment_Rate(%)"]
/var/folders/n9/29yl9jnj61q74vg3ydjdf3qh0000gn/T/ipykernel_26907/3123976544.py:16: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to a

,REF_DATE,NAICS_21,JV_industry,Industry_Label,Job Vacancies (#),Payroll Employees (#),Job Vacancy Rate (%),Employment_Rate(%),Unemployment_Rate(%),Labour_Force_Participation_Rate(%),Duration_Jobless(Months),Average_Hourly_Wage,Market_Tightness,NAICS_21_Label,Lagged_Job_Vacancy_Rate,Lagged_Unemployment_Rate,Lagged_Market_Tightness
0,2020-10-01,1,3.000,"Agriculture, forestry, fishing and hunting [11]",13630.000,188145.000,6.755,87.082,4.474,91.161,4.267,20.780,1.012,Agriculture,NaN,NaN,NaN
1,2020-10-01,2,3.000,"Agriculture, forestry, fishing and hunting [11]",2241.000,30932.000,6.755,77.169,9.718,85.476,4.811,30.582,0.466,Forestry and logging and support activities fo...,NaN,NaN,NaN
2,2020-10-01,3,3.000,"Agriculture, forestry, fishing and hunting [11]",1054.000,14543.000,6.758,50.200,29.326,71.030,4.730,27.069,0.155,"Fishing, hunting and trapping",NaN,NaN,NaN
3,2020-10-01,4,10.000,"Mining, quarrying, and oil and gas extraction ...",4195.000,183335.000,2.200,86.887,9.061,95.544,6.944,45.368,0.169,"Mining, quarrying, and oil and gas extraction",NaN,NaN,NaN
4,2020-10-01,5,17.000,Utilities [22],1420.000,122230.000,1.100,97.362,0.793,98.141,4.643,46.203,1.283,Utilities,NaN,NaN,NaN


In [29]:
monthly_industry_rates = monthly_industry_rates.merge(CPI, on=['REF_DATE'], how='left')
monthly_industry_rates.head()

,REF_DATE,NAICS_21,JV_industry,Industry_Label,Job Vacancies (#),Payroll Employees (#),Job Vacancy Rate (%),Employment_Rate(%),Unemployment_Rate(%),Labour_Force_Participation_Rate(%),Duration_Jobless(Months),Average_Hourly_Wage,Market_Tightness,NAICS_21_Label,Lagged_Job_Vacancy_Rate,Lagged_Unemployment_Rate,Lagged_Market_Tightness,Month,Year,All-items
0,2020-10-01,1,3.000,"Agriculture, forestry, fishing and hunting [11]",13630.000,188145.000,6.755,87.082,4.474,91.161,4.267,20.780,1.012,Agriculture,NaN,NaN,NaN,Oct,2020,137.500
1,2020-10-01,2,3.000,"Agriculture, forestry, fishing and hunting [11]",2241.000,30932.000,6.755,77.169,9.718,85.476,4.811,30.582,0.466,Forestry and logging and support activities fo...,NaN,NaN,NaN,Oct,2020,137.500
2,2020-10-01,3,3.000,"Agriculture, forestry, fishing and hunting [11]",1054.000,14543.000,6.758,50.200,29.326,71.030,4.730,27.069,0.155,"Fishing, hunting and trapping",NaN,NaN,NaN,Oct,2020,137.500
3,2020-10-01,4,10.000,"Mining, quarrying, and oil and gas extraction ...",4195.000,183335.000,2.200,86.887,9.061,95.544,6.944,45.368,0.169,"Mining, quarrying, and oil and gas extraction",NaN,NaN,NaN,Oct,2020,137.500
4,2020-10-01,5,17.000,Utilities [22],1420.000,122230.000,1.100,97.362,0.793,98.141,4.643,46.203,1.283,Utilities,NaN,NaN,NaN,Oct,2020,137.500


In [30]:
# 1) PREP LFS DATA: FILTER & CLEAN

# Filter LFS for employed, at-work individuals (lf_status == 1)
hours_main = LFS[LFS["lf_status"] == 1].copy()

# Keep only relevant columns
hours_main = hours_main[[
    "REF_DATE",
    "NAICS_21",
    "actualhrs_main",
    "usualhrs_main",
    "paid_overtime",
    "unpaid_overtime",
    "hrs_overtime",
    "final_wt"
]]

# Drop rows where these critical columns are NaN
hours_main.dropna(subset=[
    "actualhrs_main",
    "usualhrs_main",
    "final_wt",
    "hrs_overtime"
], inplace=True)

# Filter out rows with hrs_overtime <= 0
hours_main = hours_main[hours_main["hrs_overtime"] > 0]


# 2) CREATE HELPER COLUMNS / INDICATORS

# Main job hours difference
hours_main["main_hours_diff"] = hours_main["actualhrs_main"] - hours_main["usualhrs_main"]

# Binary: worked more than usual at the main job
hours_main["worked_more_than_usual"] = (hours_main["actualhrs_main"] > hours_main["usualhrs_main"]).astype(int)

# Binary: any unpaid overtime > 0
hours_main["any_unpaid_ot"] = (hours_main["unpaid_overtime"] > 0).astype(int)


# 3) WEIGHTED AGGREGATION FUNCTIONS

def weighted_average(df, value_col, weight_col="final_wt"):
    """Compute weighted mean of `value_col`."""
    sub = df.dropna(subset=[value_col, weight_col])
    total_weight = sub[weight_col].sum()
    if total_weight > 0:
        return (sub[value_col] * sub[weight_col]).sum() / total_weight
    return np.nan

def weighted_percentage(df, indicator_col, weight_col="final_wt"):
    """Compute weighted percentage of `indicator_col` == 1."""
    sub = df.dropna(subset=[indicator_col, weight_col])
    total_weight = sub[weight_col].sum()
    if total_weight > 0:
        return (sub[indicator_col] * sub[weight_col]).sum() / total_weight * 100
    return np.nan


# 4) GROUP & AGGREGATE

# We'll explicitly select only the columns used in the aggregator
# to avoid DeprecationWarning and ensure the grouping columns
# are not passed into the function.

# Weighted means of main_hours_diff, paid_overtime, unpaid_overtime
aggregated_main_hours = (
    hours_main.groupby(["REF_DATE", "NAICS_21"], observed=False)[["main_hours_diff","paid_overtime","unpaid_overtime","final_wt"]]
    .apply(lambda g: pd.Series({
        "avg_main_hours_diff": weighted_average(g, "main_hours_diff"),
        "avg_paid_ot_main":   weighted_average(g, "paid_overtime"),
        "avg_unpaid_ot_main": weighted_average(g, "unpaid_overtime")
    }))
    .reset_index()
)

# Weighted percentage that worked more than usual
percent_more_than_usual = (
    hours_main.groupby(["REF_DATE", "NAICS_21"], observed=False)[["worked_more_than_usual","final_wt"]]
    .apply(lambda g: weighted_percentage(g, "worked_more_than_usual"))
    .reset_index(name="percent_more_than_usual")
)

# Weighted percentage that had any unpaid OT
percent_unpaid_overtime = (
    hours_main.groupby(["REF_DATE", "NAICS_21"], observed=False)[["any_unpaid_ot","final_wt"]]
    .apply(lambda g: weighted_percentage(g, "any_unpaid_ot"))
    .reset_index(name="percent_unpaid_overtime")
)

print("aggregated_main_hours columns:", aggregated_main_hours.columns)
print(aggregated_main_hours.head())

########################################
# MERGE AGGREGATED RESULTS
########################################

# Merge average differences & OT columns
monthly_industry_rates = monthly_industry_rates.merge(
    aggregated_main_hours,
    on=["REF_DATE", "NAICS_21"],
    how="left"
)

# Convert NAICS_21 in the dataframes to the same dtype before merging
percent_more_than_usual['NAICS_21'] = percent_more_than_usual['NAICS_21'].astype(str)
percent_unpaid_overtime['NAICS_21'] = percent_unpaid_overtime['NAICS_21'].astype(str)
monthly_industry_rates['NAICS_21'] = monthly_industry_rates['NAICS_21'].astype(str)

# Merge percent_more_than_usual
monthly_industry_rates = monthly_industry_rates.merge(
    percent_more_than_usual,
    on=["REF_DATE", "NAICS_21"],
    how="left"
)

# Merge percent_unpaid_overtime
monthly_industry_rates = monthly_industry_rates.merge(
    percent_unpaid_overtime,
    on=["REF_DATE", "NAICS_21"],
    how="left"
)

# Convert NAICS_21 back to category
monthly_industry_rates['NAICS_21'] = monthly_industry_rates['NAICS_21'].astype('category')

# Quick check
print(monthly_industry_rates[[
    "REF_DATE", "NAICS_21",
    "avg_main_hours_diff", "avg_paid_ot_main", "avg_unpaid_ot_main",
    "percent_more_than_usual", "percent_unpaid_overtime"
]].head())

print("aggregated_main_hours rows:", len(aggregated_main_hours))
print("monthly_industry_rates rows:", len(monthly_industry_rates))



aggregated_main_hours columns: Index(['REF_DATE', 'NAICS_21', 'avg_main_hours_diff', 'avg_paid_ot_main', 'avg_unpaid_ot_main'], dtype='object')
    REF_DATE NAICS_21  avg_main_hours_diff  avg_paid_ot_main  avg_unpaid_ot_main
0 2020-09-01        1               14.912             9.294               8.183
1 2020-09-01        2                9.073             6.402               2.771
2 2020-09-01        3                  NaN               NaN                 NaN
3 2020-09-01        4               10.647            11.438               3.011
4 2020-09-01        5                8.269             7.054               2.354
    REF_DATE NAICS_21  avg_main_hours_diff  avg_paid_ot_main  avg_unpaid_ot_main  percent_more_than_usual  percent_unpaid_overtime
0 2020-10-01        1                9.733             7.413               6.725                   73.519                   26.768
1 2020-10-01        2                6.270            11.799               0.852                   61.533   

In [31]:

print(aggregated_main_hours.isna().sum())
print(monthly_industry_rates.isna().sum())

REF_DATE                0
NAICS_21                0
avg_main_hours_diff    15
avg_paid_ot_main       15
avg_unpaid_ot_main     15
dtype: int64
REF_DATE                               0
NAICS_21                               0
JV_industry                            0
Industry_Label                         0
Job Vacancies (#)                      0
Payroll Employees (#)                  0
Job Vacancy Rate (%)                   0
Employment_Rate(%)                     0
Unemployment_Rate(%)                   0
Labour_Force_Participation_Rate(%)     0
Duration_Jobless(Months)               0
Average_Hourly_Wage                    0
Market_Tightness                       0
NAICS_21_Label                         0
Lagged_Job_Vacancy_Rate               21
Lagged_Unemployment_Rate              21
Lagged_Market_Tightness               21
Month                                  0
Year                                   0
All-items                              0
avg_main_hours_diff                  

In [32]:
# 1) Rename Industry_Label -> JV_Industry_Label
monthly_industry_rates.rename(columns={
    "Industry_Label": "JV_Industry_Label",
    
}, inplace=True)

# 2) Remove rows where Year == 2020
monthly_industry_rates = monthly_industry_rates[monthly_industry_rates["Year"] != 2020]

print(monthly_industry_rates.isna().sum())
monthly_industry_rates.head()

REF_DATE                               0
NAICS_21                               0
JV_industry                            0
JV_Industry_Label                      0
Job Vacancies (#)                      0
Payroll Employees (#)                  0
Job Vacancy Rate (%)                   0
Employment_Rate(%)                     0
Unemployment_Rate(%)                   0
Labour_Force_Participation_Rate(%)     0
Duration_Jobless(Months)               0
Average_Hourly_Wage                    0
Market_Tightness                       0
NAICS_21_Label                         0
Lagged_Job_Vacancy_Rate                0
Lagged_Unemployment_Rate               0
Lagged_Market_Tightness                0
Month                                  0
Year                                   0
All-items                              0
avg_main_hours_diff                   11
avg_paid_ot_main                      11
avg_unpaid_ot_main                    11
percent_more_than_usual               11
percent_unpaid_o

,REF_DATE,NAICS_21,JV_industry,JV_Industry_Label,Job Vacancies (#),Payroll Employees (#),Job Vacancy Rate (%),Employment_Rate(%),Unemployment_Rate(%),Labour_Force_Participation_Rate(%),Duration_Jobless(Months),Average_Hourly_Wage,Market_Tightness,NAICS_21_Label,Lagged_Job_Vacancy_Rate,Lagged_Unemployment_Rate,Lagged_Market_Tightness,Month,Year,All-items,avg_main_hours_diff,avg_paid_ot_main,avg_unpaid_ot_main,percent_more_than_usual,percent_unpaid_overtime
63,2021-01-01,1,3.000,"Agriculture, forestry, fishing and hunting [11]",9397.000,181317.000,4.927,80.425,7.597,87.037,4.416,20.836,0.458,Agriculture,4.767,5.353,0.593,Jan,2021,139.100,7.177,2.815,5.121,86.980,58.220
64,2021-01-01,2,3.000,"Agriculture, forestry, fishing and hunting [11]",1801.000,34757.000,4.926,71.810,20.046,89.814,5.081,31.685,0.174,Forestry and logging and support activities fo...,4.767,11.216,0.283,Jan,2021,139.100,10.714,9.215,2.490,83.412,28.624
65,2021-01-01,3,3.000,"Agriculture, forestry, fishing and hunting [11]",926.000,17876.000,4.925,59.908,18.262,73.292,5.080,29.152,0.190,"Fishing, hunting and trapping",4.769,24.656,0.129,Jan,2021,139.100,12.000,12.000,0.000,100.000,0.000
66,2021-01-01,4,10.000,"Mining, quarrying, and oil and gas extraction ...",5485.000,184555.000,2.900,88.558,7.723,95.970,5.478,46.649,0.260,"Mining, quarrying, and oil and gas extraction",2.200,8.579,0.186,Jan,2021,139.100,7.988,7.462,2.879,85.933,34.818
67,2021-01-01,5,17.000,Utilities [22],1500.000,122775.000,1.200,93.794,1.845,95.557,2.880,46.301,0.594,Utilities,1.100,2.899,0.342,Jan,2021,139.100,7.163,5.825,2.019,92.587,29.539


In [33]:
monthly_industry_rates.columns
monthly_industry_rates.head()

,REF_DATE,NAICS_21,JV_industry,JV_Industry_Label,Job Vacancies (#),Payroll Employees (#),Job Vacancy Rate (%),Employment_Rate(%),Unemployment_Rate(%),Labour_Force_Participation_Rate(%),Duration_Jobless(Months),Average_Hourly_Wage,Market_Tightness,NAICS_21_Label,Lagged_Job_Vacancy_Rate,Lagged_Unemployment_Rate,Lagged_Market_Tightness,Month,Year,All-items,avg_main_hours_diff,avg_paid_ot_main,avg_unpaid_ot_main,percent_more_than_usual,percent_unpaid_overtime
63,2021-01-01,1,3.000,"Agriculture, forestry, fishing and hunting [11]",9397.000,181317.000,4.927,80.425,7.597,87.037,4.416,20.836,0.458,Agriculture,4.767,5.353,0.593,Jan,2021,139.100,7.177,2.815,5.121,86.980,58.220
64,2021-01-01,2,3.000,"Agriculture, forestry, fishing and hunting [11]",1801.000,34757.000,4.926,71.810,20.046,89.814,5.081,31.685,0.174,Forestry and logging and support activities fo...,4.767,11.216,0.283,Jan,2021,139.100,10.714,9.215,2.490,83.412,28.624
65,2021-01-01,3,3.000,"Agriculture, forestry, fishing and hunting [11]",926.000,17876.000,4.925,59.908,18.262,73.292,5.080,29.152,0.190,"Fishing, hunting and trapping",4.769,24.656,0.129,Jan,2021,139.100,12.000,12.000,0.000,100.000,0.000
66,2021-01-01,4,10.000,"Mining, quarrying, and oil and gas extraction ...",5485.000,184555.000,2.900,88.558,7.723,95.970,5.478,46.649,0.260,"Mining, quarrying, and oil and gas extraction",2.200,8.579,0.186,Jan,2021,139.100,7.988,7.462,2.879,85.933,34.818
67,2021-01-01,5,17.000,Utilities [22],1500.000,122775.000,1.200,93.794,1.845,95.557,2.880,46.301,0.594,Utilities,1.100,2.899,0.342,Jan,2021,139.100,7.163,5.825,2.019,92.587,29.539


In [36]:
monthly_rates_report = sv.analyze(monthly_rates)
monthly_rates_report.show_html('LFS_EDA_Report.html')

industry_rates_report = sv.analyze(monthly_industry_rates)
industry_rates_report.show_html('LFS_EDA_Report.html')     

                                             |          | [  0%]   00:00 -> (? left)

Report LFS_EDA_Report.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


                                             |          | [  0%]   00:00 -> (? left)

Report LFS_EDA_Report.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


In [35]:
monthly_industry_rates.to_csv('Monthly Industry Labour Market Rates.csv', index=False)